# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns),
    ]
)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), feature_columns),
    ]
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque si lo ajusto antes sobre todo el conjunto de entrenamiento, la media y la desviación estándar ya incluyen información de los folds que luego voy a usar como validación. Eso es **data leakage**: el modelo "ve" de forma indirecta datos de validación y las métricas de CV salen más optimistas de lo que serían con datos nuevos. Dentro del `Pipeline`, `cross_validate` y `GridSearchCV` reajustan el scaler en cada iteración usando únicamente los datos de entrenamiento de ese fold y solo lo aplican (`transform`) al fold de validación, que es como pasaría en producción.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([
        ("prep", clone(preprocessor_tree)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("prep", clone(preprocessor_scaled)),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("prep", clone(preprocessor_scaled)),
        ("model", KNeighborsClassifier()),
    ]),
}
list(models)

['decision_tree', 'logistic_regression', 'knn']

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    res = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring="accuracy",
        return_train_score=True,
    )
    rows.append({
        "model": name,
        "train_acc_mean": res["train_score"].mean(),
        "val_acc_mean": res["test_score"].mean(),
        "val_acc_std": res["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).set_index("model").round(4)
initial_comparison

,train_acc_mean,val_acc_mean,val_acc_std
model,,,
decision_tree,1.0000,0.9022,0.0586
logistic_regression,1.0000,0.9791,0.0277
knn,0.9771,0.9507,0.0353


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 2, 5],
    },
    "logistic_regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
    },
    "knn": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipe in models.items():
    gs = GridSearchCV(
        clone(pipe), param_grids[name],
        cv=cv, scoring="accuracy",
        return_train_score=True, n_jobs=-1,
    )
    gs.fit(X_train, y_train)
    searches[name] = gs

    best = gs.best_index_
    rows.append({
        "model": name,
        "best_cv_acc": gs.best_score_,
        "cv_std": gs.cv_results_["std_test_score"][best],
        "train_acc": gs.cv_results_["mean_train_score"][best],
        "best_params": gs.best_params_,
    })

tuned_comparison = pd.DataFrame(rows).set_index("model")
tuned_comparison["best_cv_acc"] = tuned_comparison["best_cv_acc"].round(4)
tuned_comparison["cv_std"] = tuned_comparison["cv_std"].round(4)
tuned_comparison["train_acc"] = tuned_comparison["train_acc"].round(4)
tuned_comparison

,best_cv_acc,cv_std,train_acc,best_params
model,,,,
decision_tree,0.9025,0.0762,0.9859,"{'model__max_depth': 5, 'model__min_samples_le..."
logistic_regression,0.9791,0.0277,1.0000,{'model__C': 1}
knn,0.9579,0.0405,0.9648,"{'model__n_neighbors': 7, 'model__weights': 'u..."


## 6. Selección y persistencia del campeón

In [8]:
champion_name = max(searches, key=lambda n: searches[n].best_score_)
champion = searches[champion_name].best_estimator_
print("Campeón:", champion_name)
print("Parámetros:", searches[champion_name].best_params_)

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "best_params": {k: (v if v is None or isinstance(v, (int, float, str)) else str(v))
                    for k, v in searches[champion_name].best_params_.items()},
    "cv_accuracy": float(searches[champion_name].best_score_),
    "cv_folds": 5,
    "scoring": "accuracy",
    "random_state": RANDOM_STATE,
    "features": feature_columns,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2), encoding="utf-8"
)

cv_results = pd.concat(
    [pd.DataFrame(gs.cv_results_).assign(model=name) for name, gs in searches.items()],
    ignore_index=True,
)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

Campeón: logistic_regression
Parámetros: {'model__C': 1}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Elegí **regresión logística (C = 1)** como campeón porque tuvo el mayor accuracy medio en validación cruzada (≈ 0.979) y además la menor variabilidad entre folds (desviación ≈ 0.028), frente a KNN (≈ 0.958 con k = 7 y pesos uniformes) y al árbol de decisión (≈ 0.90 con profundidad máxima 5, y una desviación de ≈ 0.076).
>
> - **Desempeño y variabilidad:** la logística es la mejor y la más estable; el árbol es el peor y el más inestable, algo esperable con solo 142 muestras de entrenamiento.
> - **Sobreajuste:** el árbol llega a ≈ 0.99–1.0 de accuracy en train pero ≈ 0.90 en validación, así que memoriza. La logística también tiene 1.0 en train, pero su accuracy de validación es casi igual de alto (0.979), por lo que la brecha es mucho menor. Lo anterior sugiere que el problema es casi linealmente separable con las variables estandarizadas, algo coherente con lo visto en el EDA (flavanoids, proline y color_intensity separan bien las clases).
> - **Complejidad e interpretabilidad:** es un modelo sencillo, rápido de entrenar y de aplicar, y sus coeficientes por clase se pueden interpretar (aunque hay variables correlacionadas, como flavanoids y total_phenols, que hay que leer con cuidado). KNN guarda todo el conjunto de entrenamiento y es más sensible a la escala.
> - **Limitación:** la diferencia entre la logística (0.979) y KNN (0.958) es de pocas muestras en un dataset pequeño, y los mismos folds se usaron para ajustar hiperparámetros y para elegir al campeón, así que la estimación de CV puede ser algo optimista. Por eso el test final (Challenge 3) es el que dará la medida honesta.

---
### Declaración de uso de IA generativa

Herramienta: Claude (Anthropic). Propósito: asistencia en programación (pipelines, validación cruzada, GridSearchCV y guardado de artefactos) y borrador de las respuestas, que contrasté con los resultados de la validación cruzada.